# Point - Facet Interactions 

## Import Packages

In [1]:
using StaticArrays 
using LinearAlgebra # provides the function norm 
using HCubature # provides adaptive numerical integration 

using Test # provides testing functionality 
using Plots 

function hcubature_count(f, a, b; kws...)
    count = 0
    i = hcubature(a, b; kws...) do x
        count += 1
        # display(count)
        f(x)
    end
    return (i..., count)
end

const Point2D = SVector{2,Float64};
const Point3D = SVector{3,Float64};

## Section 1: Introduction

Here we compute point-facet interactions. More later.

## Section 2: Computing Point - Facet Interactions

Here we compute the point (${\mathbf a}$ destination)- facet ($F$ source) interaction 

$$
P^{\phi}({\mathbf a}, F) = \iint_{F} \frac{\phi({\mathbf a})}{\| {\mathbf r} - {\mathbf a}\|} \, dS \, . 
$$
 
In case of triangular facets, $P^{\phi}$ is computed as a sum of 3 contributions, one for each edge of the triangle. This integral is singular in case that ${\mathbf a}$ lies on F. A seperate analytical solution for this case is available. 

## Section 3: Test Cases 

### First Test Case (can this be generalized to generic h values?) 

Assume 

* a triangular facet in the $xy$-plane with nodes $\vec{r}_1 = (0,0,0)$, $\vec{r}_2 = (1,0,0)$ and $\vec{r}_3 = (0,1,0)$. Then $\vec{n}_1 = (0,-1,0)$, $\vec{n}_2 = (\sqrt{2}/2,\sqrt{2}/2,0)$ and $\vec{n}_3 = (-1,0,0)$ and $R_0^1 = 0$, $R_0^2 = \sqrt{2}/2 $ and $R_0^3 = 0$. Then only the second edge contributes to the integral. Over the second edge, we have that $-\sqrt{2}/2\leq s \leq \sqrt{2}/2$. 
* a point $\vec{a}$ on the $z$-axis with coordinates $\vec{a} = (0,0,1)$. Then $\vec{\widehat{a}} = (0,0,0)$, $h=1$ and $\|  \vec{a} - \vec{r}\| = \sqrt{x^2+y^2+1}$. 
* function $\varphi(\vec{x})$ to be constant equal one, i.e., $\varphi(\vec{x}) = 1$.  

The integral to be computed can be expressed as 

$$ 
P^1(a,F) = \iint_{F} \frac{1}{\| \vec{a} - \vec{r}\|} \, dS = 
\int_0^1 \left[ \int_0^{1-x} \frac{1}{\sqrt{x^2+y^2+1}} dy \right] dx = 
- \int_0^1  \text{asinh} \left[ \frac{x}{\sqrt{x^2+1}} - \frac{1}{\sqrt{x^2+1}}\right] \, dx \, . 
$$

Numerical integration using hcubature.jl (or any alternative) gives

$$ 
P^1(a,F) = 0.437\ldots \, . 
$$

Symbolic computation instead gives (only the second edge contributes)

$$ 
P^1(a,F) = \iint_{F} \frac{1}{\| \vec{a} - \vec{r}\|} \, dS =
(\vec{r}_2 - \vec{\widehat{a}}) \cdot \vec{n}_2 \left[ I_8(s,h=1,R_0^2=\sqrt{2}/2 \right]_{s=-\sqrt{2}/2}^{s=\sqrt{2}/2} = 
\sqrt{2} / 2 \left[ I_8(s,h=1,R_0^2=\sqrt{2}/2 \right]_{s=-\sqrt{2}/2}^{s=\sqrt{2}/2} = 0.437\ldots \, . 
$$

Numerical and analytical answers do match.

## Section 2: Data Structures for Edges, EdgePairs, Facets, PointFacet Pair and Linear Functions 

How to generalize to multiple such functions? 

In [7]:
Base.@kwdef struct Facet2D 
    r1::Point2D                                # first vertex of the facet  
    r2::Point2D                                # second vertex of the facet
    r3::Point2D                                # third vertex of the facet
    tau1::Point2D = normalize(r2 - r1)         # direction of first edge   
    tau2::Point2D = normalize(r3 - r2)         # direction of second edge
    tau3::Point2D = normalize(r1 - r3)         # direction of third edge
    area::Float64 = .5*norm(cross(tau1,tau2))  # area of triangle 
    # nF::Point2D = normalize(cross(tau1,tau2))  # normal to face 
    # n1::Point2D = normalize(cross(tau1,nF))    # normal on first edge   
    # n2::Point2D = normalize(cross(tau2,nF))    # normal on second edge
    # n3::Point2D = normalize(cross(tau3,nF))    # normal on third edge
    Xmat = SMatrix{3,3,Float64, 9}(r1[1], r2[1], r3[1], r1[2], r2[2], r3[2], 1., 1., 1.) 
    rhs  = SMatrix{3,3,Float64, 9}(1I) 
    Emat = SMatrix{3,3,Float64, 9}(Xmat\rhs);
end

# https://docs.julialang.org/en/v1/manual/methods/#Function-like-objects
function (facet2D::Facet2D)(r)
    Emat = facet2D.Emat 
    phi = zeros(3)
    phi = [ Emat[1,i]*r[1] + Emat[2,i]*r[2] + Emat[3,i] for i=1:3]   
    return phi 
end 

# define function that defines the parametrization 

In [8]:
# define three vertices of a triangular facet  
r1 = Point2D(0.,0.); r2 = Point2D(1.,0.); r3 = Point2D(0.,1.);  

In [9]:
# define facet 
myFacet2D = Facet2D(r1=r1, r2=r2, r3=r3)

# define point in the facet 
a = Point2D(.5,.5)

2-element SVector{2, Float64} with indices SOneTo(2):
 0.5
 0.5

In [10]:
myFacet2D.Emat

3×3 SMatrix{3, 3, Float64, 9} with indices SOneTo(3)×SOneTo(3):
 -1.0  1.0  0.0
 -1.0  0.0  1.0
  1.0  0.0  0.0

In [11]:
myFacet2D(a)

3-element Vector{Float64}:
 0.0
 0.5
 0.5

In [7]:
Base.@kwdef struct Edge3D 
    r1::Point3D
    r2::Point3D
    len::Float64 = norm(r2-r1)
    tau::Point3D = normalize(r2-r1)
    e::Point3D = tau/len 
end 

function (edge3D::Edge3D)(r)
    return [dot(edge3D.r2-r,edge3D.e), dot(r-edge3D.r1,edge3D.e)] 
end

# here we assume lines to intersect 
function edge3Dintersection(el1::Edge3D,el2::Edge3D)
    rdiff = el2.r1-el1.r1
    num1 = dot(el1.tau,el2.tau)*dot(rdiff,el2.tau)-dot(rdiff,el1.tau)
    num2 = -dot(el1.tau,el2.tau)*dot(rdiff,el1.tau)+dot(rdiff,el2.tau)
    den  = dot(el1.tau,el2.tau)^2-1
    return [el1.r1 + (num1/den)*el1.tau, el2.r1 + (num2/den)*el2.tau]
end

# super-type to hold a pair of edges 
abstract type Edge3DPair end 

# intersecting pair of edges 
Base.@kwdef struct InterEdge3DPair <: Edge3DPair
    e1::Edge3D                                # first edge 
    e2::Edge3D                                # second edge 
    s::Point3D = edge3Dintersection(e1,e2)[1] # point of intersection 
end 

# crossing pair of edges 
Base.@kwdef struct CrossEdge3DPair <: Edge3DPair
    e1::Edge3D                                     # first edge 
    e2::Edge3D                                     # second edge 
    nw::Point3D = normalize(cross(e1.tau,e2.tau))  # normal on the plane
    h::Float64 = abs(dot(nw,e1.r1-e2.r1))          # height between the edges 
    r1hat::Point3D = e1.r1+dot(e2.r1-e1.r1,nw)*nw  # projection of start of first edge onto plane 
    r2hat::Point3D = e1.r2+dot(e2.r2-e1.r2,nw)*nw  # projection of end of first edge onto plane
    e1hat::Edge3D = Edge3D(r1 = r1hat,r2 = r2hat)  # projected first edge 
    s::Point3D = edge3Dintersection(e1hat,e2)[1]   # point of intersection 
end

# parallel pair of edges
struct ParalEdge3DPair <: Edge3DPair end

# inline pair of edges    
struct InlinEdge3DPair <: Edge3DPair end

Base.@kwdef struct Facet3D 
    r1::Point3D                                # first vertex of the facet  
    r2::Point3D                                # second vertex of the facet
    r3::Point3D                                # third vertex of the facet
    tau1::Point3D = normalize(r2 - r1)         # direction of first edge   
    tau2::Point3D = normalize(r3 - r2)         # direction of second edge
    tau3::Point3D = normalize(r1 - r3)         # direction of third edge
    area::Float64 = .5*norm(cross(tau1,tau2))  # area of triangle 
    nF::Point3D = normalize(cross(tau1,tau2))  # normal to face 
    n1::Point3D = normalize(cross(tau1,nF))    # normal on first edge   
    n2::Point3D = normalize(cross(tau2,nF))    # normal on second edge
    n3::Point3D = normalize(cross(tau3,nF))    # normal on third edge
end 

function (facet3D::Facet3D)(s)
    return 0; 
end 

Base.@kwdef struct PointFacet3DPair 
    a::Point3D                               
    F::Facet3D
    ahat::Point3D = a + dot(F.r1-a,F.nF)*F.nF
    h::Float64 = abs(dot(F.r1-a,F.nF))
    smin1::Float64 = dot(F.r1-ahat,F.tau1)
    smin2::Float64 = dot(F.r2-ahat,F.tau2)
    smin3::Float64 = dot(F.r3-ahat,F.tau3)
    smax1::Float64 = dot(F.r2-ahat,F.tau1)
    smax2::Float64 = dot(F.r3-ahat,F.tau2)
    smax3::Float64 = dot(F.r1-ahat,F.tau3)
    R01::Float64   = norm(cross(ahat-F.r1,F.tau1)) 
    R02::Float64   = norm(cross(ahat-F.r2,F.tau2))
    R03::Float64   = norm(cross(ahat-F.r3,F.tau3))
end 

Base.@kwdef struct FacetFacet3DPair 
    F1::Facet3D
    F2::Facet3D
end 

struct LinearFunction  
    a::Float64
    b::Float64    
    c::Float64
    d::Float64
end 

function (linearFunction::LinearFunction)(r)
    a = linearFunction.a
    b = linearFunction.b
    c = linearFunction.c
    d = linearFunction.d
    return [a*r[1]+linearFunction.b*r[2]+c*r[3]+d, [a,b,c]] 
end

In [8]:
# define an edge 
r1 = Point3D(1.,0.,0.,); r2 = Point3D(2.,0.,0.); 
el1 = Edge3D(r1=r1,r2=r2)

Edge3D([1.0, 0.0, 0.0], [2.0, 0.0, 0.0], 1.0, [1.0, 0.0, 0.0], [1.0, 0.0, 0.0])

In [9]:
# returns value of two basis functions associated with the edge   
s = Point3D(2.,0.,0.)
el1(s)

2-element Vector{Float64}:
 0.0
 1.0

#### Data for a Facet 

In [15]:
r1 = Point3D(0.,0.,0.); r2 = Point3D(1.,0.,0.); r3 = Point3D(0.,1.,0.);
facet = Facet3D(r1=r1, r2=r2, r3=r3) 

Facet3D([0.0, 0.0, 0.0], [1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [1.0, 0.0, 0.0], [-0.7071067811865475, 0.7071067811865475, 0.0], [0.0, -1.0, 0.0], 0.35355339059327373, [0.0, -0.0, 1.0], [0.0, -1.0, -0.0], [0.7071067811865476, 0.7071067811865476, 0.0], [-1.0, 0.0, 0.0])

In [16]:
facet.tau1

3-element SVector{3, Float64} with indices SOneTo(3):
 1.0
 0.0
 0.0

#### Data for PointFacet3D Pair 

In [17]:
a = Point3D(0.,0.,10.);
r1 = Point3D(0.,0.,0.); r2 = Point3D(1.,0.,0.); r3 = Point3D(0.,1.,0.);
facet = Facet3D(r1=r1, r2=r2, r3=r3)
pointfacetpair = PointFacet3DPair(a = a, F = facet) 

PointFacet3DPair([0.0, 0.0, 10.0], Facet3D([0.0, 0.0, 0.0], [1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [1.0, 0.0, 0.0], [-0.7071067811865475, 0.7071067811865475, 0.0], [0.0, -1.0, 0.0], 0.35355339059327373, [0.0, -0.0, 1.0], [0.0, -1.0, -0.0], [0.7071067811865476, 0.7071067811865476, 0.0], [-1.0, 0.0, 0.0]), [0.0, 0.0, 0.0], 10.0, 0.0, -0.7071067811865475, -1.0, 1.0, 0.7071067811865475, 0.0, 0.0, 0.7071067811865475, 0.0)

In [18]:
#[pointfacetpair.R01, pointfacetpair.R02, pointfacetpair.R03] 
pointfacetpair.smin2

-0.7071067811865475

#### Linear Function Testing 

In [19]:
phi = LinearFunction(1.,2.,3.,4.)
r = Point3D(1.,-1.,1.,)
display(phi(r)[1]) 
display(phi(r)[2])

6.0

3-element Vector{Float64}:
 1.0
 2.0
 3.0

## Section 10: Point - Facet Interactions

In [1]:
function P1(pointfacet3Dpair::PointFacet3DPair,linearfunction::LinearFunction)

    ahat  = pointfacet3Dpair.ahat
    h     = pointfacet3Dpair.h 
    F     = pointfacet3Dpair.F 
    smin1 = pointfacet3Dpair.smin1
    smin2 = pointfacet3Dpair.smin2 
    smin3 = pointfacet3Dpair.smin3 
    smax1 = pointfacet3Dpair.smax1
    smax2 = pointfacet3Dpair.smax2 
    smax3 = pointfacet3Dpair.smax3 
    R01   = pointfacet3Dpair.R01 
    R02   = pointfacet3Dpair.R02
    R03   = pointfacet3Dpair.R03 

    phival  = linearfunction(ahat)[1]    
    phigrad = linearfunction(ahat)[2]

    t1 = 0.   
    if !(R01==0.) 
        t1  = dot(F.r1-smin1*F.tau1-ahat,phigrad)*I10(smax1,h,R01)
        t1 += h*dot(F.tau1,phigrad)*I11(smax1,h,R01)+phival*I8(smax1,h,R01) 
        t1 -= dot(F.r1-smin1*F.tau1-ahat,phigrad)*I10(smin1,h,R01)
        t1 -= h*dot(F.tau1,phigrad)*I11(smin1,h,R01)+phival*I8(smin1,h,R01)
        t1 *= dot(F.r1-ahat,F.n1)
    end 

    t2 = 0.   
    if !(R02==0.) 
        t2  = dot(F.r2-smin2*F.tau2-ahat,phigrad)*I10(smax2,h,R02)
        t2 += h*dot(F.tau2,phigrad)*I11(smax2,h,R02)+phival*I8(smax2,h,R02)  
        t2 -= dot(F.r2-smin2*F.tau2-ahat,phigrad)*I10(smin2,h,R02)
        t2 -= h*dot(F.tau2,phigrad)*I11(smin2,h,R02)+phival*I8(smin2,h,R02)     
        t2 *= dot(F.r2-ahat,F.n2)
    end 

    t3 = 0.
    if !(R02==0.)
        t3  = dot(F.r3-smin2*F.tau3-ahat,phigrad)*I10(smax3,h,R03)
        t3 += h*dot(F.tau3,phigrad)*I11(smax3,h,R03)+phival*I8(smax3,h,R03)  
        t3 -= dot(F.r3-smin3*F.tau3-ahat,phigrad)*I10(smin3,h,R03)
        t3 -= h*dot(F.tau3,phigrad)*I11(smin3,h,R03)+phival*I8(smin3,h,R03)     
        t3 *= dot(F.r3-ahat,F.n3)
    end 
    
    return t1+t2+t3
end

function P1phiconst(pointfacet3Dpair::PointFacet3DPair,linearfunction::LinearFunction)

    ahat  = pointfacet3Dpair.ahat
    h     = pointfacet3Dpair.h 
    F     = pointfacet3Dpair.F 
    smin1 = pointfacet3Dpair.smin1
    smin2 = pointfacet3Dpair.smin2 
    smin3 = pointfacet3Dpair.smin3 
    smax1 = pointfacet3Dpair.smax1
    smax2 = pointfacet3Dpair.smax2 
    smax3 = pointfacet3Dpair.smax3 
    R01   = pointfacet3Dpair.R01 
    R02   = pointfacet3Dpair.R02
    R03   = pointfacet3Dpair.R03 

    t1 = 0.; 
    if !(R01==0.) 
        t1  = I8(smax1,h,R01)-I8(smin1,h,R01)  
        t1 *= dot(F.r1-ahat,F.n1)
    end 

    t2 = 0.;
    if !(R02==0.)
       t2  = I8(smax2,h,R02)-I8(smin2,h,R02)         
       t2 *= dot(F.r2-ahat,F.n2)
    end 

    t3 = 0.;
    if !(R03==0.)
        t3  = I8(smax3,h,R03)-I8(smin3,h,R03)
        t3 *= dot(F.r3-ahat,F.n3)
    end 
    
    return t1+t2+t3
end

function P1integrand(pointfacetpair::PointFacet3DPair)
    a = pointfacetpair.a
    F = pointfacetpair.F  
    return 0.
end

LoadError: UndefVarError: `PointFacet3DPair` not defined

In [34]:
# test special case of constant function 
a = Point3D(0.25,.25,.25);
r1 = Point3D(0.,0.,0.); r2 = Point3D(1.,0.,0.); r3 = Point3D(0.,1.,0.);
facet = Facet3D(r1=r1, r2=r2, r3=r3)
pointfacetpair = PointFacet3DPair(a = a, F = facet);
phi = LinearFunction(0.,0.,0.,1.)
@test P1(pointfacetpair,phi) ≈ P1phiconst(pointfacetpair,phi)

Test Passed

In [35]:
a = Point3D(0.,0.,1.);
r1 = Point3D(0.,0.,0.); r2 = Point3D(1.,0.,0.); r3 = Point3D(0.,1.,0.);
facet = Facet3D(r1=r1, r2=r2, r3=r3)
pointfacetpair = PointFacet3DPair(a = a, F = facet)
phi = LinearFunction(0.,0.,0.,1.)
P1phiconst(pointfacetpair,phi)
# P1(pointfacetpair,phi)

0.4369992897579717

In [68]:
pointfacetpair.F.tau3

3-element SVector{3, Float64} with indices SOneTo(3):
  0.0
 -1.0
  0.0

In [64]:
pointfacetpair.R02

0.7071067811865475

In [56]:
pointfacetpair.smax2

0.7071067811865475

In [52]:
pointfacetpair.F.n2

3-element SVector{3, Float64} with indices SOneTo(3):
 0.7071067811865476
 0.7071067811865476
 0.0

In [70]:
I8(pointfacetpair.smin2,pointfacetpair.h,pointfacetpair.R02)

-0.30900516116156673

In [71]:
sqrt(2)/2*(I8(pointfacetpair.smax2,pointfacetpair.h,pointfacetpair.R02) - I8(pointfacetpair.smin2, pointfacetpair.h,pointfacetpair.R02))

0.4369992897579717

In [73]:
a

3-element SVector{3, Float64} with indices SOneTo(3):
 0.0
 0.0
 1.0

In [72]:
# attempt to compute Pphi numerically  
hcubature(x -> x[1]/sqrt((x[1]-a[1])^2+(x[1]*x[2]-a[2])^2+a[3]^2), (0,0), (1,1))

(0.39667956065749077, 5.569539877903207e-9)

In [77]:
# does evaluating I8 for h=0 pose annty challenges? 
I8(1,0,1)

0.881373587019543